## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [1]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"; MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model


# 12 · Inference walkthrough and project handoff

This notebook loads the selected artifact and scores an already-engineered row. It is an offline walkthrough, not a real-time service: behavioral features need historical state and must use only events available before a transaction.

### Why load an engineered row, not a raw CSV row

The saved estimator was trained on the feature matrix created in notebook 04. Scoring raw fields directly would omit velocity, recency, history, category encoding, and missingness transformations. This walkthrough uses a precomputed competition row to demonstrate the model contract; a real-time feature service still needs point-in-time history.

In [ ]:
artifact_path = MODEL_DIR / "best_experiment_model.joblib"
feature_path = INTERMEDIATE_DIR / "test_features.pkl"
if not artifact_path.exists() or not feature_path.exists():
    raise FileNotFoundError("Run notebooks 04 and 06 first.")

artifact = joblib.load(artifact_path)
test_features = pd.read_pickle(feature_path)
feature_cols = list(artifact["feature_cols"])
missing_features = sorted(set(feature_cols) - set(test_features.columns))
if missing_features:
    raise ValueError(f"Test data lacks required inputs: {missing_features[:10]}")

print(f"Loaded {type(artifact['model']).__name__}")
print(f"Required model features: {len(feature_cols)}")

## 1. Score one competition example

### Why preserve the one-row DataFrame shape

`iloc[[0]]` returns one row while retaining the two-dimensional table shape expected by scikit-learn and XGBoost. The imputer and scaler, when present, must run in the same order and with the same fitted statistics as training. The resulting score is model output, not a final approve/decline decision.

In [ ]:
row = test_features.iloc[[0]]
X_row = row[feature_cols].replace([np.inf, -np.inf], np.nan)

# Apply the saved preprocessing in the same order as during training.
if artifact.get("imputer") is not None:
    X_row = artifact["imputer"].transform(X_row)
if artifact.get("scaler") is not None:
    X_row = artifact["scaler"].transform(X_row)

probability = float(artifact["model"].predict_proba(X_row)[0, 1])
result = {
    "transaction_id": row["TransactionID"].iloc[0]
    if "TransactionID" in row else None,
    "model_probability_output": probability,
    "risk_score_0_to_100": round(100 * probability, 2),
    "model_name": artifact.get(
        "experiment_name", type(artifact["model"]).__name__
    ),
}
result

## 2. Safe inference contract

A real service should validate required inputs, reject or explicitly handle invalid values, record a request id and model version, and return documented decision information. Malformed input must not silently become a low-risk score. The precomputed row is only a demonstration of the expected model matrix.

### Why validate inputs before inference

The checks expose missing required columns and non-finite numeric values before they reach the estimator. In a service, schema validation should fail clearly or follow a documented fallback; silently filling unexpected missing inputs could turn a data outage into a misleading low-risk score.

In [ ]:
numeric_inputs = row[feature_cols].select_dtypes(include=[np.number])
missing_required = set(feature_cols) - set(row.columns)
non_finite_count = int((~np.isfinite(numeric_inputs.to_numpy())).sum())

print(f"Required model features: {len(feature_cols)}")
print(f"Missing required columns: {len(missing_required)}")
print(f"Non-finite numeric values: {non_finite_count}")

## 3. Notebook path and handoff

- **01–03:** data ingestion, validation, exploration.
- **04:** time-aware feature engineering.
- **05–06:** baseline and model experiments.
- **07–08:** evaluation and risk scoring.
- **09–10:** explanations and decision policy.
- **11–12:** monitoring and an offline inference walkthrough.

The production handoff still requires a point-in-time feature pipeline, a later untouched temporal test set, a service API, access controls, and operational review. These notebooks do not claim to provide a live fraud system.